# Homework 1: Agentic RAG

First, ensure we have the library installed, I use uv and run it on my terminal:
```python
uv add jupyter gitsource openai python-dotenv requests minsearch toyaikit
```

You can also use pip in this notebook, for example:
```python
!pip install gitsource
```

In [1]:
# Test notebook
print("Hello World!")

Hello World!


## Q1. How many lesson pages

In [2]:
# Initialize the reader to fetch markdown files from the /lessons/ folders
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [f.parse() for f in files]

In [3]:
# Check one raw filename
print(files[0].filename)

01-agentic-rag/lessons/01-intro.md


In [4]:
# Check modules name
modules = set(d["filename"].split("/")[0] for d in documents)
print(modules)

{'04-evaluation', '06-best-practices', '05-monitoring', '01-agentic-rag', '07-project-example', '03-orchestration', '02-vector-search'}


In [5]:
# Count the number of lesson pages (documents)
print(f"Number of lesson pages: {len(documents)}")

Number of lesson pages: 72


## Q2. Indexing and searching

In [6]:
# Import library
import minsearch

In [7]:
print(documents[0].keys())

dict_keys(['content', 'filename'])


In [9]:
# Index the documents with the specified schema
index = minsearch.Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [10]:
# Perform the search
query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query, num_results=5)

In [14]:
# Review the results
for r in search_results:
    print(r["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md


In [15]:
# Extract the filename of the first result
print(f"First result filename: {search_results[0]['filename']}")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

In [17]:
# For linux user
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

# For Windows user
!curl -o rag_helper.py https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0   0     0   0     0     0     0  --:--:-- --:--:-- --:--:--     0
  0     0   0     0   0     0     0     0  --:--:-- --:--:-- --:--:--     0
100  2134 100  2134   0     0  2296     0  --:--:-- --:--:-- --:--:--  2302


In [18]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [22]:
# Update the original `RAGBase` class 
from rag_helper import RAGBase
import dataclasses

@dataclasses.dataclass
class RAGResult:
    answer: str
    usage: object

class LessonRAG(RAGBase):

    def search(self, query, num_results=5):
        # no boost_dict/filter_dict here — your index has no
        # 'question'/'section'/'course' fields to boost or filter on
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"FILE: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        return response  # full response object now, not just .output_text

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return RAGResult(answer=response.output_text, usage=response.usage)

In [24]:
rag = LessonRAG(index=index, llm_client=openai_client)
result = rag.rag("How does the agentic loop keep calling the model until it stops?")
print(result.usage)

ResponseUsage(input_tokens=7126, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7238)


## Q4. Chunking

In [25]:
# Import library
from gitsource import chunk_documents

# Apply chunking with a 2000-character window and 1000-character step
# This will split the 72 pages into smaller pieces
chunks = chunk_documents(documents, size=2000, step=1000)

# Count the total number of resulting chunks
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


In [26]:
# Manual calculation
total_chars = sum(len(d['content']) for d in documents)
print(total_chars, total_chars / 1000)  # rough chunks-per-step estimate

332534 332.534


In [28]:
# Calculation testing on one document
doc = documents[0]

from gitsource.chunking import sliding_window

test_chunks = sliding_window(doc['content'], size=2000, step=1000)
print(len(doc['content']), len(test_chunks))

3183 3


## Q5. RAG with chunking

In [29]:
# Check chunk keys
print(chunks[0].keys())

dict_keys(['start', 'content', 'filename'])


In [30]:
# Index the chunks in minsearch
# We treat the chunk objects as dictionaries for the index
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)

# Initialize the RAG assistant with the chunk index
# Using the modified LessonRAG class from Q3
chunk_rag = LessonRAG(index=chunk_index, llm_client=openai_client)
result_q5 = chunk_rag.rag("How does the agentic loop keep calling the model until it stops?")
print(result_q5.usage)

ResponseUsage(input_tokens=2309, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=96, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2405)


In [35]:
in_token_q3 = result.usage.input_tokens
in_token_q5 = result_q5.usage.input_tokens

print(f"Initial Input Tokens: {in_token_q3}")
print(f"Chunked Input Tokens: {in_token_q5}")
print(f"Input Tokens Ratio Before vs After Chunked: {in_token_q3//in_token_q5}x")

Initial Input Tokens: 7126
Chunked Input Tokens: 2309
Input Tokens Ratio Before vs After Chunked: 3x


## Q6. Turning it into an agent

In [43]:
# Import Library
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [44]:
# Define the search tool
from typing import List, Dict

def search(query: str) -> List[Dict]:
    """Search the course lesson chunks for relevant passages matching the query."""
    return chunk_index.search(
        query, 
        num_results=5
    )

In [45]:
# Register the tool
tools = Tools()
tools.add_tool(search)
tools.get_tools()  # confirm the generated schema looks sane — check the exact method name if this errors

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lesson chunks for relevant passages matching the query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [46]:
# Add a call counter to the search tool
# framework-agnostic — works no matter which agent library we actually end up using
search_call_count = {"count": 0}

def search(query: str) -> List[Dict]:
    """Search the course lesson chunks for relevant passages matching the query."""
    search_call_count["count"] += 1
    return chunk_index.search(query, num_results=5)

In [47]:
# Register the tool
tools = Tools()
tools.add_tool(search)
tools.get_tools()  # confirm the generated schema looks sane — check the exact method name if this errors

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lesson chunks for relevant passages matching the query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [48]:
# Build the Agent (toyaikit path)
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
openai_client = OpenAIClient(model="gpt-5.4-mini", client=OpenAI())

instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=openai_client,
)

In [49]:
# Run the agentic query
query = "How does the agentic loop work, and how is it different from plain RAG?"
result = runner.loop(prompt=query, callback=callback)

-> Response received


-> Response received


In [51]:
# Read the counts
print(f"Total search calls: {search_call_count["count"]}")

Total search calls: 3


Thanks !!!